# Classificazione Dataset con Modello AI (Streaming Edition)

Questo notebook è ottimizzato per **non esaurire la RAM**. 
1. Estrae una cache di progetti unici in parallelo (leggendo solo i campi strettamente necessari).
2. Classifica i progetti unici a blocchi (batch).
3. Associa le classificazioni e salva i file, elaborandoli uno per uno tramite ThreadPool, in modo che in RAM ci siano solo pochi file contemporaneamente.

In [1]:
import os
import glob
import pandas as pd
from multiprocessing import Pool
from traceability_classification_worker import get_unique_projects

INPUT_DIR = '../../data/technology_mapping'
file_pattern = os.path.join(INPUT_DIR, 'reclassified_multiclass_*.csv')
files = glob.glob(file_pattern)
print(f"Trovati {len(files)} file.")

NUM_THREADS = 15  # Come richiesto, limite a 15 thread/processi

# 1. Creazione della Cache Globale in modalità Memory-Efficient
print(f"Estrazione progetti unici in parallelo (max {NUM_THREADS} processi)...")
with Pool(processes=NUM_THREADS) as pool:
    cache_parts = pool.map(get_unique_projects, files)

# Filtra i risultati nulli e concatena
cache_parts = [p for p in cache_parts if p is not None]
cache_df = pd.concat(cache_parts, ignore_index=True)

# Dedup finale basata SOLO sulla descrizione (per evitare duplicati con titoli leggermente diversi)
cache_df = cache_df.drop_duplicates(subset=['DESCRIZIONE_PROGETTO']).reset_index(drop=True)

print(f"Cache costruita! Trovati {len(cache_df)} progetti unici reali.")


Trovati 0 file.
Estrazione progetti unici in parallelo (max 15 processi)...


ValueError: No objects to concatenate

In [ ]:
import requests
import math
from tqdm.auto import tqdm

# 2. Classificazione della Cache via API
API_URL = "http://localhost:8080/classify"
BATCH_SIZE = 512  # Invia 512 record alla volta

predictions_labels = []
predictions_confidences = []
predictions_positive_prob = []  # P(tracciabilita): permette di ri-tarare la soglia a valle

num_batches = math.ceil(len(cache_df) / BATCH_SIZE)
print(f"Inizio classificazione: {len(cache_df)} record in {num_batches} batch da {BATCH_SIZE}...")

# Sessione persistente per non sovraccaricare le connessioni TCP
session = requests.Session()

with tqdm(total=len(cache_df), desc="Record classificati", unit="rec") as pbar:
    for i in range(num_batches):
        batch = cache_df.iloc[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        
        texts = []
        for _, row in batch.iterrows():
            # Stesso formato del training (dataset.py:load_dataset_from_csv):
            # "titolo: descrizione", con fallback alla sola descrizione se il titolo manca
            # (niente "nan: ..." quando TITOLO_PROGETTO è NaN).
            titolo = '' if pd.isna(row['TITOLO_PROGETTO']) else str(row['TITOLO_PROGETTO']).strip()
            desc = '' if pd.isna(row['DESCRIZIONE_PROGETTO']) else str(row['DESCRIZIONE_PROGETTO']).strip()
            texts.append(f"{titolo}: {desc}" if titolo else desc)
            
        payload = {"texts": texts}
        
        try:
            response = session.post(API_URL, json=payload, timeout=120)
            response.raise_for_status()
            results = response.json().get('predictions', [])
            for res in results:
                predictions_labels.append(res['label'])
                predictions_confidences.append(res['confidence'])
                predictions_positive_prob.append(res.get('positive_prob'))
        except Exception as e:
            print(f"Errore nel batch {i}: {e}")
            for _ in range(len(texts)):
                predictions_labels.append(None)
                predictions_confidences.append(None)
                predictions_positive_prob.append(None)
                
        pbar.update(len(batch))

cache_df['AI_LABEL'] = predictions_labels
cache_df['AI_CONFIDENCE'] = predictions_confidences
cache_df['AI_POSITIVE_PROB'] = predictions_positive_prob
print("Classificazione completata!")


/home/gabs/Documenti/Università/AI nelle Imprese/open-data-analytics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Inizio classificazione: 929962 record in 1817 batch da 512...



Record classificati:   0%|          | 0/929962 [00:00<?, ?rec/s]


Record classificati:   0%|          | 512/929962 [00:00<23:16, 665.41rec/s]


Record classificati:   0%|          | 1024/929962 [00:01<22:22, 691.82rec/s]

...


Record classificati: 100%|█████████▉| 929280/929962 [20:39<00:01, 504.76rec/s]


Record classificati: 100%|█████████▉| 929792/929962 [20:40<00:00, 503.52rec/s]


Record classificati: 100%|██████████| 929962/929962 [20:40<00:00, 487.10rec/s]


Record classificati: 100%|██████████| 929962/929962 [20:40<00:00, 749.45rec/s]

Classificazione completata!


In [ ]:
from concurrent.futures import ThreadPoolExecutor

# 3. Associazione e Salvataggio (Streaming per risparmiare RAM)
OUTPUT_DIR = '../../data/classified_technology_mapping'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Riduciamo la cache al minimo indispensabile per il join
cache_for_merge = cache_df[['DESCRIZIONE_PROGETTO', 'AI_LABEL', 'AI_CONFIDENCE', 'AI_POSITIVE_PROB']]

def process_and_save_file(filepath):
    try:
        # Legge un singolo file
        df = pd.read_csv(filepath)
        
        # Join per aggiungere le colonne di classificazione basandoci sulla DESCRIZIONE
        merged_df = df.merge(
            cache_for_merge, 
            on='DESCRIZIONE_PROGETTO', 
            how='left'
        )
        
        # Salva il file
        output_path = os.path.join(OUTPUT_DIR, os.path.basename(filepath))
        merged_df.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Errore nel file {filepath}: {e}")
        return False

print(f"Salvataggio streaming in corso (max {NUM_THREADS} thread contemporanei)...")
# Usiamo ThreadPoolExecutor così i thread condividono cache_for_merge senza duplicarla in memoria
with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    results = list(tqdm(executor.map(process_and_save_file, files), total=len(files), desc="File processati"))

success_count = sum(1 for r in results if r)
print(f"Completato! {success_count}/{len(files)} file elaborati e salvati correttamente.")


Salvataggio streaming in corso (max 15 thread contemporanei)...



File processati:   0%|          | 0/12 [00:00<?, ?it/s]

/tmp/ipykernel_56031/1808615952.py:13: DtypeWarning: Columns (0: COD_STRUMENTI, 1: CLASSIFICAZIONE_MULTICLASS, 2: TIPO_AI, 3: TECNOLOGIE_AI) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


/tmp/ipykernel_56031/1808615952.py:13: DtypeWarning: Columns (0: COD_STRUMENTI, 1: CLASSIFICAZIONE_MULTICLASS, 2: TIPO_AI, 3: TECNOLOGIE_AI) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


/tmp/ipykernel_56031/1808615952.py:13: DtypeWarning: Columns (0: COD_STRUMENTI, 1: CLASSIFICAZIONE_MULTICLASS, 2: TIPO_AI, 3: TECNOLOGIE_AI) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


...


File processati:   8%|▊         | 1/12 [01:58<21:46, 118.79s/it]


File processati:  17%|█▋        | 2/12 [02:30<11:13, 67.32s/it] 


File processati:  25%|██▌       | 3/12 [03:27<09:23, 62.61s/it]


File processati: 100%|██████████| 12/12 [03:27<00:00, 17.26s/it]

Completato! 12/12 file elaborati e salvati correttamente.
